In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

BASE_DIR = "dataset"
IMG_SIZE = (224, 224)
BATCH_SIZE = 8

# Derive training dataset
train_ds = image_dataset_from_directory(
    BASE_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=67
)

# Derive validation dataset
val_ds = image_dataset_from_directory(
    BASE_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=67
)

# Save class names
# Preprocessing gets rid of them in dataset metadata
class_names = train_ds.class_names

# Preprocess x (images) in dataset to MobileNetV2 model
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

Found 197 files belonging to 2 classes.
Using 158 files for training.
Found 197 files belonging to 2 classes.
Using 39 files for validation.


In [3]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load MobileNetV2 (no top layer)
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights='imagenet')

# Freeze convolutional base
base_model.trainable = False

# Functional instead of sequential (better serialization when saving)
inputs = tf.keras.Input(shape=(224, 224, 3)) # Input tensor
x = base_model(inputs, training=False) # Pass input through base model (high-level features)
x = layers.GlobalAveragePooling2D()(x) # Convert feature map to single vector per image
x = layers.Dense(128, activation='relu')(x) # Add layer to learn abstract representations
x = layers.Dropout(0.3)(x) # Dropout for regularization (prevent overfitting)
outputs = layers.Dense(len(class_names), activation='softmax')(x) # Output layer for classification

model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,210 (9.24 MB)

 Trainable params: 164,226 (641.51 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
# Train model (5 epochs)
history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=5)

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.7292 - loss: 0.5913 - val_accuracy: 1.0000 - val_loss: 0.0066
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9937 - loss: 0.0191 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 1.0000 - loss: 0.0048 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.0029 - val_accuracy: 1.0000 - val_loss: 9.0901e-04
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 4.3456e-04


In [7]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

# Test model prediction
img_path = "dataset/neutral/100.jpg"
img = load_img(img_path, target_size=IMG_SIZE)
x = img_to_array(img) # Convert image to tensor
x = preprocess_input(x) # Preprocess for MobileNetV2
x = np.expand_dims(x, axis=0) # Add batch dimension (model expects batches)

pred = model.predict(x)
print("Raw predictions:", pred)
print("Predicted class:", class_names[np.argmax(pred)]) # Index of highest probability

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Raw predictions: [[4.1160826e-04 9.9958837e-01]]
Predicted class: neutral


In [8]:
model.save('nailong_exp_model.keras')